In [0]:
# Load Tables

from pyspark.sql.functions import col, to_date

proc_df = spark.table("medical_project.gold.fact_procedures")
date_df = spark.table("medical_project.gold.dim_date")

display(proc_df)

In [0]:
# Join with Date Dimension

proc_with_date = proc_df.join(
    date_df,
    proc_df.procedure_date == date_df.date,
    "left"
)

In [0]:
# Create Half-Year Column

from pyspark.sql.functions import when

proc_with_date = proc_with_date.withColumn(
    "half_year",
    when(col("month") <= 6, "H1").otherwise("H2")
)

display(proc_with_date)

In [0]:
# Aggregate Metrics

from pyspark.sql.functions import avg, count

agg_df = proc_with_date.groupBy(
    "year", "half_year", "procedure_description"
).agg(
    avg("base_cost").alias("avg_cost"),
    count("*").alias("procedure_count")
)

In [0]:
# Rank Top 10 per Half-Year

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("year", "half_year") \
                    .orderBy(col("avg_cost").desc())

ranked_df = agg_df.withColumn(
    "rank",
    row_number().over(window_spec)
)

In [0]:
# Filter Top 10

kpi4 = ranked_df.filter(
    col("rank") <= 10
).select(
    "year",
    "half_year",
    "procedure_description",
    "avg_cost",
    "procedure_count"
).orderBy("year", "half_year", "rank")

display(kpi4)

In [0]:
# Save Table 

kpi4.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_top_procedures")